# Dong-Included Baseline XGBoost + SHAP Colab

This notebook reflects the latest feedback:

- The study is about **regional differences by `읍면동`**, not a pure time-series prediction problem.
- Therefore, `읍면동` must be included in the baseline feature set.
- The main comparison is now:
  - `A_location_baseline`: housing/contract variables + `읍면동`
  - `B_location_spatial`: `A_location_baseline` + sea/park/school spatial-derived variables
- Dong-level correlations are added as EDA to show neighborhood-level relationships.

## 1. Final Experiment Logic

| Experiment | Role | Features |
|---|---|---|
| `A_location_baseline` | Main baseline | apartment structure + contract time + `읍면동` |
| `B_location_spatial` | Main spatial model | baseline + sea/park/school spatial variables |

Why this design:

- `읍면동` is a core regional variable, so excluding it from baseline weakens the study logic.
- Spatial-derived variables are tested as **additional neighborhood information beyond administrative dong identity**.
- Because most spatial variables are dong-level variables, the improvement after adding them may be small. That is expected and should be interpreted as a conservative test.

## 2. Colab Setup

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import importlib.util

REPO_URL = "https://github.com/hyeon03-sketch/IML-Final-project.git"
REPO_BRANCH = "codex/ml-project-review"
REPO_DIR = Path("IML-Final-project")
DATA_FILE = Path("IML_Final_dataset.xlsx")

if not DATA_FILE.exists() and Path("../IML_Final_dataset.xlsx").exists():
    os.chdir("..")

if not DATA_FILE.exists():
    if not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL])
    os.chdir(REPO_DIR)

print("Working directory:", Path.cwd())
print("Dataset exists:", Path("IML_Final_dataset.xlsx").exists())

packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "openpyxl": "openpyxl",
    "sklearn": "scikit-learn",
    "xgboost": "xgboost",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "shap": "shap",
}
missing = [pip_name for import_name, pip_name in packages.items() if importlib.util.find_spec(import_name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Installed:", missing)
else:
    print("All required packages are already installed.")

## 3. Imports and Korean Font Setup

In [ ]:
import warnings
from pathlib import Path
import subprocess

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import shap

from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupShuffleSplit, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
RND = 42
DATA_PATH = Path("IML_Final_dataset.xlsx")
SHEET_NAME = "최종데이터셋_모델용"
TARGET = "전세환산보증금(만원)"
CONVERSION_RATE = 0.065
OUT_DIR = Path("outputs_dong_baseline_xgboost_shap")
OUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)


def setup_korean_font():
    candidate_paths = [
        Path("/usr/share/fonts/truetype/nanum/NanumGothic.ttf"),
        Path("/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf"),
        Path("/System/Library/Fonts/AppleSDGothicNeo.ttc"),
        Path("/Library/Fonts/AppleGothic.ttf"),
    ]
    if not any(p.exists() for p in candidate_paths):
        try:
            subprocess.check_call(["apt-get", "update", "-qq"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            subprocess.check_call(["apt-get", "install", "-y", "fonts-nanum"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        except Exception as exc:
            print("Korean font installation skipped:", exc)
    for cache_file in Path.home().glob(".cache/matplotlib/fontlist*"):
        try:
            cache_file.unlink()
        except Exception:
            pass
    font_paths = [p for p in candidate_paths if p.exists()]
    for font_path in font_paths:
        try:
            fm.fontManager.addfont(str(font_path))
        except Exception:
            pass
    try:
        fm._load_fontmanager(try_read_cache=False)
    except Exception:
        pass
    for font_path in font_paths:
        try:
            font_name = fm.FontProperties(fname=str(font_path)).get_name()
            mpl.rcParams["font.family"] = [font_name]
            mpl.rcParams["font.sans-serif"] = [font_name]
            mpl.rcParams["axes.unicode_minus"] = False
            sns.set_theme(font=font_name)
            print("Font:", font_name)
            return font_name
        except Exception:
            continue
    mpl.rcParams["axes.unicode_minus"] = False
    print("Font: default")
    return "default"

setup_korean_font()

## 4. Load Data and Build Target

Monthly-rent rows are retained by converting monthly rent into Jeonse-equivalent deposits.

$$
\text{Jeonse-equivalent deposit} = \text{Deposit} + \frac{\text{Monthly rent} \times 12}{0.065}
$$

In [ ]:
def make_jeonse_equivalent_target(data, conversion_rate=CONVERSION_RATE):
    out = data.copy()
    deposit = pd.to_numeric(out["보증금(만원)"], errors="coerce")
    monthly_rent = pd.to_numeric(out["월세금(만원)"], errors="coerce").fillna(0)
    lease_type = out["전월세구분"].astype(str).str.strip()
    target = deposit.astype(float).copy()
    monthly_mask = lease_type.eq("월세")
    target.loc[monthly_mask] = deposit.loc[monthly_mask] + monthly_rent.loc[monthly_mask] * 12.0 / conversion_rate
    out[TARGET] = target
    return out


def first_mode(series):
    modes = series.mode(dropna=False)
    return modes.iloc[0] if len(modes) else np.nan


def harmonize_dong_level_features(data, feature_cols, group_col="읍면동"):
    out = data.copy()
    rows = []
    for col in feature_cols:
        unique_by_dong = out.groupby(group_col)[col].nunique(dropna=False)
        inconsistent = unique_by_dong[unique_by_dong > 1]
        before = out[col].copy()
        out[col] = out.groupby(group_col)[col].transform(first_mode)
        changed = int((before.astype(str) != out[col].astype(str)).sum())
        rows.append({
            "feature": col,
            "inconsistent_dong_count_before": len(inconsistent),
            "rows_changed": changed,
            "max_unique_values_within_dong_after": int(out.groupby(group_col)[col].nunique(dropna=False).max()),
        })
    return out, pd.DataFrame(rows)

DONG_LEVEL_COLS = [
    "바다여부", "공원여부", "공원수", "최대공원면적(㎡)", "총공원면적(㎡)",
    "동_초등학교수", "동_중학교수", "동_고등학교수", "동_총학교수", "동_초중고모두있음여부",
]

raw = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
df = make_jeonse_equivalent_target(raw)
df = df[df[TARGET].notna() & df[TARGET].gt(0)].copy()
df, harmonization_report = harmonize_dong_level_features(df, DONG_LEVEL_COLS)

print("Original rows:", len(raw))
print("Modeling rows:", len(df))
print("Jeonse rows:", int(df["전월세구분"].eq("전세").sum()))
print("Monthly-rent rows converted:", int(df["전월세구분"].eq("월세").sum()))
print("Dong count:", df["읍면동"].nunique())
print("Target mean:", round(df[TARGET].mean(), 2))
print("Target median:", round(df[TARGET].median(), 2))

display(harmonization_report)
harmonization_report.to_csv(OUT_DIR / "dong_level_harmonization_report.csv", index=False, encoding="utf-8-sig")

## 5. Dong-Level EDA: Neighborhood Differences and Correlations

This section is added for the presentation feedback: show neighborhood-level relationships because the analysis is organized by `읍면동`.

Important: these are descriptive correlations across dongs, not causal effects.

In [ ]:
SPATIAL_FEATURES_FOR_EDA = [
    "바다여부", "공원여부", "공원수", "최대공원면적(㎡)",
    "동_초등학교수", "동_중학교수", "동_고등학교수", "동_초중고모두있음여부",
]

dong_summary = (
    df.groupby("읍면동")
    .agg(
        거래수=(TARGET, "size"),
        평균_전세환산보증금=(TARGET, "mean"),
        중앙값_전세환산보증금=(TARGET, "median"),
        평균_전용면적=("전용면적(㎡)", "mean"),
        평균_건물연령=("건물연령(계약기준)", "mean"),
        평균_층=("층", "mean"),
        **{col: (col, "first") for col in SPATIAL_FEATURES_FOR_EDA}
    )
    .reset_index()
    .sort_values("평균_전세환산보증금", ascending=False)
)

display(dong_summary)
dong_summary.to_csv(OUT_DIR / "dong_summary.csv", index=False, encoding="utf-8-sig")

plt.figure(figsize=(10, 7))
plot_df = dong_summary.sort_values("평균_전세환산보증금", ascending=True)
plt.barh(plot_df["읍면동"], plot_df["평균_전세환산보증금"], color="#2563eb")
plt.xlabel("평균 전세환산보증금(만원)")
plt.ylabel("읍면동")
plt.title("읍면동별 평균 전세환산보증금")
plt.tight_layout()
plt.savefig(OUT_DIR / "dong_mean_target_bar.png", dpi=150)
plt.show()

corr_cols = [
    "평균_전세환산보증금", "중앙값_전세환산보증금", "거래수",
    "평균_전용면적", "평균_건물연령", "평균_층",
    "바다여부", "공원여부", "공원수", "최대공원면적(㎡)",
    "동_초등학교수", "동_중학교수", "동_고등학교수", "동_초중고모두있음여부",
]
dong_corr = dong_summary[corr_cols].corr(numeric_only=True)
display(dong_corr)
dong_corr.to_csv(OUT_DIR / "dong_level_correlation.csv", encoding="utf-8-sig")

plt.figure(figsize=(11, 9))
sns.heatmap(dong_corr, annot=True, fmt=".2f", cmap="vlag", center=0, square=False)
plt.title("읍면동 단위 변수 상관관계")
plt.tight_layout()
plt.savefig(OUT_DIR / "dong_level_correlation_heatmap.png", dpi=150)
plt.show()

## 6. Main Feature Sets

The baseline includes `읍면동`, following the latest feedback.

In [ ]:
BASE_FEATURES = ["전용면적(㎡)", "층", "건물연령(계약기준)", "계약연도", "계약월"]
DONG_FEATURE = ["읍면동"]
SPATIAL_FEATURES = [
    "바다여부",
    "공원여부",
    "공원수",
    "최대공원면적(㎡)",
    "동_초등학교수",
    "동_중학교수",
    "동_고등학교수",
    "동_초중고모두있음여부",
]

FEATURE_GROUPS = {
    "A_location_baseline": BASE_FEATURES + DONG_FEATURE,
    "B_location_spatial": BASE_FEATURES + DONG_FEATURE + SPATIAL_FEATURES,
}

BINARY_COLS = {"바다여부", "공원여부", "동_초중고모두있음여부"}
CATEGORICAL_COLS = {"읍면동"}

for name, cols in FEATURE_GROUPS.items():
    print(name, len(cols), cols)

## 7. XGBoost Implementation

In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def split_feature_types(cols):
    categorical = [c for c in cols if c in CATEGORICAL_COLS]
    binary = [c for c in cols if c in BINARY_COLS]
    numeric = [c for c in cols if c not in set(categorical + binary)]
    return numeric, categorical, binary


def make_preprocessor(cols):
    numeric, categorical, binary = split_feature_types(cols)
    transformers = []
    pass_cols = numeric + binary
    if pass_cols:
        transformers.append(("pass", "passthrough", pass_cols))
    if categorical:
        transformers.append(("cat", make_one_hot_encoder(), categorical))
    return ColumnTransformer(transformers=transformers, remainder="drop")


def make_xgboost():
    return XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        random_state=RND,
        n_jobs=-1,
        importance_type="gain",
    )

XGB_PARAM_GRID = {
    "model__n_estimators": [300, 600],
    "model__max_depth": [4, 6],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.9],
    "model__colsample_bytree": [0.9],
}


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mape_pct(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true > 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def evaluate_predictions(y_true, pred):
    return {
        "test_rmse": rmse(y_true, pred),
        "test_mae": float(mean_absolute_error(y_true, pred)),
        "test_mape_pct": mape_pct(y_true, pred),
        "test_r2": float(r2_score(y_true, pred)),
    }


def fit_xgb_grid(data, group_name, cols, train_idx, test_idx):
    X = data[cols].copy()
    y = data[TARGET].copy()
    X_train, X_test = X.loc[train_idx], X.loc[test_idx]
    y_train, y_test = y.loc[train_idx], y.loc[test_idx]

    pipe = Pipeline([("pre", make_preprocessor(cols)), ("model", make_xgboost())])
    search = GridSearchCV(
        pipe,
        param_grid=XGB_PARAM_GRID,
        scoring="neg_root_mean_squared_error",
        cv=KFold(n_splits=5, shuffle=True, random_state=RND),
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
    pred = search.predict(X_test)
    metrics = evaluate_predictions(y_test, pred)
    metrics.update({
        "experiment": group_name,
        "model": "XGBoost",
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        "cv_rmse": float(-search.best_score_),
        "best_params": search.best_params_,
    })
    fitted = {"estimator": search.best_estimator_, "X_test": X_test, "y_test": y_test, "pred": pred}
    return metrics, fitted

## 8. Main Model Comparison

In [ ]:
train_idx, test_idx = train_test_split(df.index, test_size=0.2, random_state=RND)

rows = []
fitted = {}
for group_name, cols in FEATURE_GROUPS.items():
    print("Running", group_name)
    metrics, bundle = fit_xgb_grid(df, group_name, cols, train_idx, test_idx)
    rows.append(metrics)
    fitted[group_name] = bundle
    print(f"  RMSE={metrics['test_rmse']:.2f}, MAE={metrics['test_mae']:.2f}, MAPE={metrics['test_mape_pct']:.2f}%, R2={metrics['test_r2']:.4f}")

results = pd.DataFrame(rows).sort_values("test_rmse")
display(results[["experiment", "model", "n_train", "n_test", "cv_rmse", "test_rmse", "test_mae", "test_mape_pct", "test_r2", "best_params"]])
results.to_csv(OUT_DIR / "main_xgboost_results.csv", index=False, encoding="utf-8-sig")

a = results[results["experiment"].eq("A_location_baseline")].iloc[0]
b = results[results["experiment"].eq("B_location_spatial")].iloc[0]
delta = pd.DataFrame([{
    "comparison": "B_location_spatial - A_location_baseline",
    "delta_rmse": b["test_rmse"] - a["test_rmse"],
    "delta_mae": b["test_mae"] - a["test_mae"],
    "delta_mape_pct": b["test_mape_pct"] - a["test_mape_pct"],
    "delta_r2": b["test_r2"] - a["test_r2"],
}])
delta["rmse_improved"] = delta["delta_rmse"] < 0
delta["mape_improved"] = delta["delta_mape_pct"] < 0
delta["r2_improved"] = delta["delta_r2"] > 0
display(delta)
delta.to_csv(OUT_DIR / "main_xgboost_delta.csv", index=False, encoding="utf-8-sig")

## 9. Prediction vs Actual

In [ ]:
for group_name in FEATURE_GROUPS:
    bundle = fitted[group_name]
    plt.figure(figsize=(6, 6))
    plt.scatter(bundle["y_test"], bundle["pred"], s=10, alpha=0.45, color="#2563eb")
    lim = max(bundle["y_test"].max(), np.max(bundle["pred"]))
    plt.plot([0, lim], [0, lim], "r--", lw=1)
    plt.xlabel("Actual Jeonse-equivalent deposit (만원)")
    plt.ylabel("Predicted Jeonse-equivalent deposit (만원)")
    plt.title(f"Prediction vs Actual: {group_name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"prediction_vs_actual_{group_name}.png", dpi=150)
    plt.show()

## 10. SHAP for `B_location_spatial`

This is the main interpretability model because it includes both `읍면동` and spatial-derived variables.

In [ ]:
SHAP_SAMPLE_SIZE = 800
FINAL_GROUP = "B_location_spatial"


def feature_group(feature_name):
    clean = str(feature_name).split("__", 1)[-1]
    if clean.startswith("읍면동_") or "읍면동" in clean:
        return "dong_location"
    if "바다" in clean:
        return "sea"
    if "공원" in clean:
        return "park"
    if "학교" in clean or "초중고" in clean:
        return "school"
    if "전용면적" in clean or clean == "층" or "건물연령" in clean:
        return "housing_structure"
    if clean in {"계약연도", "계약월"}:
        return "contract_time"
    return "other"

bundle = fitted[FINAL_GROUP]
pipe = bundle["estimator"]
pre = pipe.named_steps["pre"]
model = pipe.named_steps["model"]

X_test = bundle["X_test"].copy()
X_shap = X_test.sample(min(len(X_test), SHAP_SAMPLE_SIZE), random_state=RND)
X_transformed = pre.transform(X_shap)
feature_names = pre.get_feature_names_out()

explainer = shap.TreeExplainer(model)
raw_shap_values = explainer.shap_values(X_transformed)
shap_values = raw_shap_values.values if hasattr(raw_shap_values, "values") else raw_shap_values

shap_importance = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)
shap_importance["feature_group"] = shap_importance["feature"].map(feature_group)

group_shap = (
    shap_importance.groupby("feature_group", as_index=False)["mean_abs_shap"]
    .sum()
    .sort_values("mean_abs_shap", ascending=False)
)
group_shap["share_pct"] = group_shap["mean_abs_shap"] / group_shap["mean_abs_shap"].sum() * 100
spatial_share = group_shap[group_shap["feature_group"].isin(["school", "park", "sea"])] ["share_pct"].sum()

print(f"Spatial SHAP share (school + park + sea): {spatial_share:.2f}%")
display(shap_importance.head(30))
display(group_shap)

shap_importance.to_csv(OUT_DIR / "shap_feature_importance_location_spatial.csv", index=False, encoding="utf-8-sig")
group_shap.to_csv(OUT_DIR / "shap_group_importance_location_spatial.csv", index=False, encoding="utf-8-sig")

plt.figure(figsize=(9, 7))
top = shap_importance.head(20).iloc[::-1]
plt.barh(top["feature"], top["mean_abs_shap"], color="#059669")
plt.title("SHAP Feature Importance: B_location_spatial")
plt.xlabel("Mean absolute SHAP value")
plt.tight_layout()
plt.savefig(OUT_DIR / "shap_top20_location_spatial.png", dpi=150)
plt.show()

plt.figure(figsize=(7, 4))
sns.barplot(data=group_shap, y="feature_group", x="share_pct", color="#2563eb")
plt.title("Grouped SHAP Importance Share")
plt.xlabel("Share of total mean absolute SHAP (%)")
plt.ylabel("")
plt.tight_layout()
plt.savefig(OUT_DIR / "shap_group_importance_location_spatial.png", dpi=150)
plt.show()

## 11. Strict Validation

The main comparison remains `B_location_spatial - A_location_baseline`.

In [ ]:
RUN_STRICT_VALIDATION = True


def run_holdout(validation_name, train_idx, test_idx):
    rows = []
    for group_name, cols in FEATURE_GROUPS.items():
        print("Strict", validation_name, group_name)
        metrics, _ = fit_xgb_grid(df, group_name, cols, train_idx, test_idx)
        metrics["validation"] = validation_name
        rows.append(metrics)
    return rows

if RUN_STRICT_VALIDATION:
    strict_rows = []
    train_idx_s, test_idx_s = train_test_split(df.index, test_size=0.2, random_state=RND)
    strict_rows.extend(run_holdout("random_split", train_idx_s, test_idx_s))

    years = sorted(df["계약연도"].dropna().unique())
    if len(years) >= 2:
        latest_year = years[-1]
        train_idx_s = df.index[df["계약연도"] < latest_year]
        test_idx_s = df.index[df["계약연도"] == latest_year]
        strict_rows.extend(run_holdout(f"time_holdout_test_{latest_year}", train_idx_s, test_idx_s))

    complex_groups = df["단지명"].fillna("missing_complex")
    train_pos, test_pos = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RND).split(df, groups=complex_groups))
    strict_rows.extend(run_holdout("complex_holdout", df.index[train_pos], df.index[test_pos]))

    dong_groups = df["읍면동"].fillna("missing_dong")
    train_pos, test_pos = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RND).split(df, groups=dong_groups))
    strict_rows.extend(run_holdout("dong_holdout", df.index[train_pos], df.index[test_pos]))

    strict_results = pd.DataFrame(strict_rows).sort_values(["validation", "experiment"])
    display(strict_results[["validation", "experiment", "n_train", "n_test", "cv_rmse", "test_rmse", "test_mae", "test_mape_pct", "test_r2", "best_params"]])
    strict_results.to_csv(OUT_DIR / "strict_validation_results.csv", index=False, encoding="utf-8-sig")

    delta_rows = []
    for validation in strict_results["validation"].unique():
        sub = strict_results[strict_results["validation"].eq(validation)]
        a = sub[sub["experiment"].eq("A_location_baseline")].iloc[0]
        b = sub[sub["experiment"].eq("B_location_spatial")].iloc[0]
        delta_rows.append({
            "validation": validation,
            "delta_rmse": b["test_rmse"] - a["test_rmse"],
            "delta_mae": b["test_mae"] - a["test_mae"],
            "delta_mape_pct": b["test_mape_pct"] - a["test_mape_pct"],
            "delta_r2": b["test_r2"] - a["test_r2"],
        })
    strict_delta = pd.DataFrame(delta_rows)
    strict_delta["spatial_improves_rmse"] = strict_delta["delta_rmse"] < 0
    strict_delta["spatial_improves_mape"] = strict_delta["delta_mape_pct"] < 0
    strict_delta["spatial_improves_r2"] = strict_delta["delta_r2"] > 0
    display(strict_delta)
    strict_delta.to_csv(OUT_DIR / "strict_validation_delta.csv", index=False, encoding="utf-8-sig")
else:
    print("Strict validation skipped")

## 12. Reporting Text

Use this wording in the report:

> 본 연구는 포항시 북구 전세가격을 읍면동 단위의 지역적 특성과 함께 분석하는 것을 목적으로 하므로, `읍면동`을 baseline 모델의 독립변수로 포함하였다. 이후 바다 접근성, 공원, 학교 등 공간파생변수를 추가하여, 행정동 정보만으로 설명되지 않는 지역 환경 정보가 예측 성능을 보완하는지 검증하였다.

> 읍면동 단위 EDA에서는 동별 평균 전세환산보증금과 공원·학교·바다 관련 변수의 상관관계를 확인하였다. 이 분석은 인과관계 검정이 아니라, 지역별 전세가격 차이와 공간환경 변수 사이의 관계를 기술적으로 보여주기 위한 것이다.